### GRASS setup


In [ ]:
import os, sys, shutil, subprocess, re, glob 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GRASSBIN = "/usr/local/bin/grass85"
GISDBASE = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks"
py_path  = subprocess.check_output([GRASSBIN, "--config", "python_path"], text=True).strip()
dist_dir = str(Path(py_path).parents[1])  # .../dist.x86_64-pc-linux-gnu

env = os.environ.copy()
env["GISBASE"] = dist_dir                      # ✅ IMPORTANT
env["PATH"] = f"{dist_dir}/bin:{dist_dir}/scripts:" + env["PATH"]
env["PYTHONPATH"] = py_path + os.pathsep + env.get("PYTHONPATH","")
env["LD_LIBRARY_PATH"] = f"{dist_dir}/lib:" + env.get("LD_LIBRARY_PATH","")

os.environ.update(env)
sys.path.insert(0, py_path)

import grass.script as gs
import grass.jupyter as gj



### Create New project

In [ ]:
#gs.create_project("DEM_generation", epsg=3358, overwrite=True)

In [ ]:
#gj.init(GISDBASE, location="DEM_generation", mapset="PERMANENT", grass_path=GRASSBIN)

### Create new mapset

In [ ]:
#gs.run_command("g.mapset", flags="c", mapset="DTM_DSM", overwrite=True)

In [ ]:
gj.init(GISDBASE, location="DEM_generation", mapset="DTM_DSM", grass_path=GRASSBIN)

In [ ]:
gs.run_command("r.mask", raster="hull_mask", overwrite=True)

### Import 2017 lidar DTM (Ground+Road)

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DTM_ground_water_road")


In [ ]:
folder = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DTM_ground_water_road"
tif_files = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]

imported_rasters = []
for tif in tif_files:
    full_path = os.path.join(folder, tif)
    raster_name = os.path.splitext(tif)[0]
    gs.run_command(
        "r.import",
        input=full_path,
        output=raster_name,
        resample="bilinear",   # for orthos; use "nearest" for categorical
        overwrite=True
    )
    imported_rasters.append(raster_name)

print(imported_rasters)

In [ ]:
gs.run_command("g.region", raster=",".join(imported_rasters), res=1)

In [ ]:
gs.run_command("r.patch",
               input=",".join(reversed(imported_rasters)),
               output="lidar_2017_DTM",
               overwrite=True)


In [ ]:
gs.run_command("r.relief", input="lidar_2017_DTM", output="lidar_2017_DTM_relief")

In [ ]:
m = gj.Map()
m.d_shade(color="lidar_2017_DTM", shade="lidar_2017_DTM_relief", brighten=50)
#m.d_rast(map="lidar_2017_DTM")
m.d_legend(raster="lidar_2017_DTM", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

### Import 2017 lidar DSM

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DSM_allclassified")

In [ ]:
folder = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DSM_allclassified"
tif_files = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]

imported_DSM_rasters = []
for tif in tif_files:
    full_path = os.path.join(folder, tif)
    raster_name = os.path.splitext(tif)[0]
    gs.run_command(
        "r.import",
        input=full_path,
        output=raster_name,
        resample="bilinear",   # for orthos; use "nearest" for categorical
        overwrite=True
    )
    imported_DSM_rasters.append(raster_name)

print(imported_DSM_rasters)


In [ ]:
gs.run_command("r.patch",
               input=",".join(reversed(imported_DSM_rasters)),
               output="lidar_2017_DSM",
               overwrite=True)


In [ ]:
gs.run_command("r.relief", input="lidar_2017_DSM", output="lidar_2017_DSM_relief")

In [ ]:
m = gj.Map()
m.d_shade(color="lidar_2017_DSM", shade="lidar_2017_DSM_relief", brighten=50)
#m.d_rast(map="lidar_2017_DTM")
m.d_legend(raster="lidar_2017_DSM", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

### Data import lidar 2020

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/open_topo_2020/Rasters")

In [ ]:
!g.region raster="lidar_2017_DSM" 

In [ ]:
gs.run_command(
    "r.import",
    input="rasters_NC21Scheip/output_be.tif",  
    output="lidar_2020_DTM",
    resolution="region",         
    resample="bilinear",
    overwrite=True)

In [ ]:
gs.run_command("r.relief", input="lidar_2020_DTM", output="lidar_2020_DTM_relief")


In [ ]:
#!g.region raster=lidar_2017_DSM save=saved_region

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="lidar_2020_DTM", shade="lidar_2020_DTM_relief", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
m.d_legend(raster="lidar_2020_DTM", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

In [ ]:
gs.run_command(
    "r.import",
    input="rasters_NC21Scheip/output_hh.tif",  
    output="lidar_2020_DSM",
    resolution="region",         
    resample="bilinear",
    overwrite=True)

In [ ]:
gs.run_command("r.relief", input="lidar_2020_DSM", output="lidar_2020_DSM_relief")

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="lidar_2020_DSM", shade="lidar_2020_DSM_relief", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
m.d_legend(raster="lidar_2020_DSM", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

#### Canopy height model from DSM and DTM

In [ ]:
gs.run_command(
    "r.import",
    input="CHM/CHM.tif",  
    output="lidar_2020_CHM",
    resolution="region",         
    resample="bilinear",
    overwrite=True)

In [ ]:
gs.run_command("r.relief", input="lidar_2020_CHM", output="lidar_2020_CHM_relief")

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="lidar_2020_CHM", shade="lidar_2020_CHM_relief", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
m.d_legend(raster="lidar_2020_CHM", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

### Data Import SFM 2024(Just after helene)

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Aerial_photogrammetry")

In [ ]:
!g.region raster="lidar_2017_DSM" 

##### Import and create DSM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="A0069A_data_upload_chimney_sfm_med_point_cloud.copc.laz",
    output="sfm_pc_DSM",
    #class_filter="2,11",  # Ground and Roads
    overwrite=True
)

In [ ]:
gs.run_command(
    "v.surf.rst",
    input="sfm_pc_DSM",
    elevation="sfm_dsm",
    smooth=1,
    dmin=1,
    segmax=40,
    tension=20,
    npmin=120, 
    overwrite=True
)

In [ ]:
    gs.run_command(
        "r.fillnulls",
        input="sfm_dsm",
        output="sfm_dsm_filled",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="sfm_dsm_filled", output="sfm_DSM_relief_filled")

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="sfm_dsm_filled", shade="sfm_DSM_relief_filled", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
#m.d_rast(map="sfm_DSM_relief")
m.d_legend(raster="sfm_dsm_filled", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

##### Import ground and road for DTM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="A0069A_data_upload_chimney_sfm_med_point_cloud.copc.laz",
    output="sfm_DTM",
    class_filter="2,11",  # Ground and Roads
    overwrite=True
)

In [ ]:
## used corey's parameters
gs.run_command(
    "v.surf.rst",
    input="sfm_DTM",
    elevation="sfm_dtm",
    smooth=1,
    dmin=1,
    segmax=40,
    tension=20,
    npmin=120,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="sfm_dtm",
        output="sfm_dtm_filled",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
## parameters suggested by helena
gs.run_command(
    "v.surf.rst",
    input="sfm_DTM",
    elevation="sfm_dtm_hel",
    smooth=1,
    dmin=2,
    segmax=40,
    tension=20,
    npmin=300,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="sfm_dtm_hel",
        output="sfm_dtm_hel_filled",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="sfm_dtm_filled", output="sfm_dtm_relief_filled")

In [ ]:
gs.run_command("r.relief", input="sfm_dtm_hel_filled", output="sfm_dtm_hel_relief_filled")

In [ ]:
m = gj.Map(use_region=True)
#m.d_shade(color="sfm_dtm", shade="sfm_dtm_relief", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
m.d_rast(map="sfm_dtm_relief_filled")
#m.d_legend(raster="sfm_dtm", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
#m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

In [ ]:
m = gj.Map(use_region=True)
m.d_rast(map="sfm_dtm_hel_relief_filled")
m.show()

### Data Import lidar 2024 (After helene from will)

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/After_helene_lidar")

##### Import dtm data and create DTM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="lidar_2024_dtm_points_epsg3358.las",
    output="lidar_2024_dtm",
    flags="wr",  
    overwrite=True
)

In [ ]:
gs.run_command(
    "v.hull",
    input="lidar_2024_dtm",
    output="hull",
    flags="f",
    quiet=True
)

In [ ]:
gs.run_command(
    "v.to.rast",
    input="hull",
    output="hull_mask",
    use="value",
    type="area",
    value=1
)

In [ ]:
gs.run_command(
    "v.surf.rst",
    input="lidar_2024_dtm",
    elevation="lidar_2024_dtm",
    smooth=1,
    dmin=2,
    segmax=40,
    tension=20,
    npmin=300,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="lidar_2024_dtm",
        output="lidar_2024_dtm_filled",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="lidar_2024_dtm_filled", output="lidar_2024_dtm_relief_filled")

In [ ]:
m = gj.Map()
m.d_rast(map="lidar_2024_dtm_relief_filled")
m.show()

##### Import dsm data and create DSM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="lidar_2024_dsm_points_epsg3358.las",
    output="lidar_2024_dsm",
    flags="wr",
    overwrite=True
)

In [ ]:
gs.run_command(
    "v.surf.rst",
    input="lidar_2024_dsm",
    elevation="lidar_2024_dsm",
    smooth=1,
    dmin=2,
    segmax=40,
    tension=20,
    npmin=300,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="lidar_2024_dsm",
        output="lidar_2024_dsm_filled",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="lidar_2024_dsm_filled", output="lidar_2024_dsm_relief_filled")

In [ ]:
m = gj.Map(use_region=True)
m.d_rast(map="lidar_2024_dsm_relief_filled")
m.show()

### Import orthophotos of all years , 2018,2020,2022, 2024

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Orthos")

In [ ]:
!g.region raster=lidar_2017_DTM

In [ ]:
ortho_dir = Path("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Orthos")
files = sorted(ortho_dir.glob("*.tif"))

if not files:
    raise FileNotFoundError(f"No .tif files found in: {ortho_dir}")

imported = []
for f in files:
    name = f.stem
    print(f"Importing: {name}")

    gs.run_command(
        "r.import",            # reprojects into current GRASS location if needed
        input=str(f),
        output=name,
        resample="bilinear",   # good default for orthophotos
        overwrite=True
    )
    imported.append(name)

print(f"Done. Imported {len(imported)} rasters.")

In [ ]:
import re
import grass.script as gs

def list_rasters(pattern):
    txt = gs.read_command("g.list", type="raster", pattern=pattern).strip()
    return [x for x in txt.splitlines() if x]

def raster_exists(name):
    ff = gs.find_file(name, element="cell")
    return bool(ff.get("name"))

def pick_rgb(base):
    maps = list_rasters(f"{base}*")
    # common cases from r.import: base.1 base.2 base.3 (and sometimes .4)
    dot = []
    for m in maps:
        mm = re.match(rf"^{re.escape(base)}\.(\d+)$", m)
        if mm:
            dot.append((int(mm.group(1)), m))
    dot = [m for _, m in sorted(dot)]
    if len(dot) >= 3:
        return dot[0], dot[1], dot[2]

    # sometimes: base_1 base_2 base_3
    under = []
    for m in maps:
        mm = re.match(rf"^{re.escape(base)}_(\d+)$", m)
        if mm:
            under.append((int(mm.group(1)), m))
    under = [m for _, m in sorted(under)]
    if len(under) >= 3:
        return under[0], under[1], under[2]

    # sometimes: base.red base.green base.blue
    named = {m.split(".")[-1].lower(): m for m in maps if m.startswith(base + ".")}
    if {"red", "green", "blue"} <= set(named):
        return named["red"], named["green"], named["blue"]

    raise RuntimeError(f"Could not find 3 band rasters for '{base}'. Found: {maps}")

# ---- make composites ----
composites = []
for base in imported:   # 'imported' is your list from the r.import loop
    print(f"\nBase: {base}")
    print("Available rasters:", list_rasters(f"{base}*"))

    r, g, b = pick_rgb(base)
    out = f"{base}_rgb"
    print(f"Using bands -> R:{r}, G:{g}, B:{b}")
    print(f"Creating    -> {out}")

    gs.run_command("r.composite", red=r, green=g, blue=b, output=out, overwrite=True)

    if raster_exists(out):
        print(f"✅ Created GRASS raster map: {out}")
        composites.append(out)
    else:
        print(f"❌ r.composite ran, but {out} not found. Check messages above.")

print("\nAll composites:", composites)


In [ ]:
gs.run_command(
    "r.patch",
    input="Ortho_NAIP_2024_1,Ortho_NAIP_2024_2",
    output="Ortho_NAIP_2024",
    overwrite=True
)

In [ ]:
# gs.run_command("r.series", 
#                input="sfm_dtm,lidar_2017_DTM,lidar_2020_DTM,lidar_2024_dtm",
#                output="dsm_count",
#                method="count")


In [ ]:
# !r.mapcalc "dsm_overlap = if(dsm_count == 4, 1, null())"

In [ ]:
#!r.to.vect input=dsm_overlap output=dsm_overlap_poly type=area


In [ ]:
#!v.clean input=dsm_overlap_poly output=dsm_overlap_poly_clean tool=rmarea thresh=1000


In [ ]:
map = gj.InteractiveMap()
#map.add_raster("Ortho_aerial_2024_rgb")
map.add_raster("lidar_2024_dsm_relief_filled@DTM_DSM")
map.add_vector("new_profile")
map.add_vector("stable_points")
#map.add_vector("dsm_overlap_poly")
map.show()

### Generate profile before vertical correction



In [ ]:
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks")

In [ ]:
gj.init(GISDBASE, location="DEM_generation", mapset="PERMANENT", grass_path=GRASSBIN)

In [ ]:
!g.list type=vector

In [ ]:
map = gj.InteractiveMap()
#map.add_raster("Ortho_aerial_2024_rgb")
map.add_raster("lidar_2024_dsm_relief_filled@DTM_DSM")
#map.add_vector("new_profile")
map.add_vector("stable_points")
#map.add_vector("dsm_overlap_poly")
map.show()

In [ ]:
line_vec = "profile_test5"   # <-- change this

rasters = [
    "lidar_2017_DTM","lidar_2017_DSM",
    "lidar_2020_DTM","lidar_2020_DSM",
    "lidar_2024_dtm_filled","lidar_2024_dsm_filled",
    "sfm_dtm_filled","sfm_dsm_filled",
]

# Export ALL features as WKT and parse ALL vertices (fixes the "always same line" issue)
wkt_txt = gs.read_command("v.out.ascii", input=line_vec, format="wkt").strip()
pairs = re.findall(r"(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)", wkt_txt)

if len(pairs) < 2:
    raise RuntimeError(f"Could not extract coordinates from {line_vec}")

# Remove consecutive duplicate vertices (optional but helps)
coords_list = []
prev = None
for x, y in pairs:
    if prev != (x, y):
        coords_list.append(f"{x},{y}")
        prev = (x, y)

coords = ",".join(coords_list)

print("Vertices:", len(coords_list))
print("First:", coords_list[0], " Last:", coords_list[-1])

out_dir = Path(f"./profiles_out_{line_vec}")
out_dir.mkdir(parents=True, exist_ok=True)

for r in rasters:
    out_csv = out_dir / f"profile_{r}.csv"
    gs.run_command(
        "r.profile",
        input=r,
        coordinates=coords,
        resolution=1,
        output=str(out_csv),
        flags="g",
        overwrite=True,
    )
    print("Wrote:", out_csv)

print("Done ->", out_dir)

In [ ]:
line_vec = "profile_test5"  # must match above
files = sorted(glob.glob(f"./profiles_out_{line_vec}/profile_*.csv"))
if not files:
    raise FileNotFoundError("No profile CSVs found for this line_vec.")

def load_profile(f):
    df = pd.read_csv(f, header=None, comment="#", sep=None, engine="python")
    if df.shape[1] == 1:
        df = df[0].astype(str).str.strip()
        df = df[(df != "") & (~df.str.startswith("#"))].str.split(r"[,\s]+", expand=True)
    df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")

    if df.shape[1] >= 3:
        x, y, v = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,-1].to_numpy()
        d = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        return d, v
    return df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy()

plt.figure(figsize=(18, 6))
for f in files:
    d, v = load_profile(f)
    label = re.sub(r"^profile_|\.csv$", "", os.path.basename(f))
    plt.plot(d, v, label=label)

plt.xlabel("Distance (m)")
plt.ylabel("Elevation")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)

plt.savefig(f"profile_plot_{line_vec}.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
m = gj.InteractiveMap(width=1500)
m.add_raster("Ortho_NAIP_2024_1_rgb@DTM_DSM") 
m.add_raster("Ortho_NAIP_2024_2_rgb@DTM_DSM") 
m.add_vector("stable_points@DTM_DSM")
m.show()

### We can see clearly that there is some vertical shift in lidar 2024 DSM and DTM, Now we need to correct that.
We will take 2020 lidar data as reference and correct all according to that. We will use stable road points for the correction.

In [ ]:
!g.list type=raster

In [ ]:
# points = "stable_points"          # <-- your vector
# r_ref  = "lidar_2020_DTM"         # reference raster
# r_tgt  = "lidar_2024_dtm"         # raster you want to compare/correct

# col_ref = "z_2020"
# col_tgt = "z_2024"

# # 1) ensure attribute columns exist (skip if they already exist)
# for col in (col_ref, col_tgt):
#     try:
#         gs.run_command("v.db.addcolumn", map=points,
#                        columns=f"{col} DOUBLE PRECISION")
#     except gs.CalledModuleError:
#         pass

# # 2) sample rasters into the point table
# gs.run_command("v.what.rast", map=points, raster=r_ref, column=col_ref)
# gs.run_command("v.what.rast", map=points, raster=r_tgt, column=col_tgt)



In [ ]:
from pandas import json_normalize

PTS   = "stable_points"
R_REF = "lidar_2020_DTM_ref"

TARGETS = [
    "lidar_2017_DTM",
    "lidar_2017_DSM",
    "lidar_2024_dtm_filled",
    "lidar_2024_dsm_filled",
    "sfm_dtm_filled",
    "sfm_dsm_filled",
]

SUFFIX = "_vref2020_rmdse"

def signed_rmdse_bias(dem1: str, dem2: str, control_points: str):
    data = gs.parse_command(
        "r.what",
        flags="n",
        map=f"{dem2},{dem1}",
        points=control_points,
        format="json",
    )
    df = json_normalize(data)

    dem1_elev = f"{dem1}.value"
    dem2_elev = f"{dem2}.value"

    # make numeric + drop missing
    df[dem1_elev] = np.array(df[dem1_elev], dtype=float)
    df[dem2_elev] = np.array(df[dem2_elev], dtype=float)
    df = df[np.isfinite(df[dem1_elev]) & np.isfinite(df[dem2_elev])].copy()

    diff = df[dem2_elev] - df[dem1_elev]            # target - reference
    med  = float(np.median(diff))                   # SIGN comes from this
    mag  = float(np.sqrt(np.median(diff**2)))       # RMdSE magnitude (always +)

    sgn = 1.0 if med >= 0 else -1.0                 # if target above -> +, below -> -
    signed_bias = sgn * mag                         # signed RMdSE shift
    return signed_bias, mag, med

def outname(raster, suffix):
    return f"{raster.split('@')[0]}{suffix}"

gs.run_command("g.region", raster=R_REF)

results = []
for tgt in TARGETS:
    out = outname(tgt, SUFFIX)

    signed_bias, mag, med = signed_rmdse_bias(R_REF, tgt, PTS)
    # apply: corrected = target - signed_bias
    #  - if target above (med>0): subtract +mag
    #  - if target below (med<0): subtract -mag  => add mag
    gs.mapcalc(f"{out} = {tgt} - ({signed_bias})", overwrite=True)

    new_signed_bias, new_mag, new_med = signed_rmdse_bias(R_REF, out, PTS)

    print(f"\n--- {tgt} -> {out} ---")
    print(f"median(diff)={med:.4f}  RMdSE_mag={mag:.4f}  signed_bias={signed_bias:.4f}")
    print(f"after: median(diff)={new_med:.4f}  RMdSE_mag={new_mag:.4f}  signed_bias={new_signed_bias:.4f}")

    results.append((tgt, out, signed_bias, mag, med, new_mag, new_med))

print("\nDONE.")

#### Now new profile after correction

In [ ]:
#g.list type=vector

In [ ]:
rename_pairs = {
    "lidar_2017_DTM_vref2020_rmdse":          "lidar_2017_DTM_corr",
    "lidar_2017_DSM_vref2020_rmdse":          "lidar_2017_DSM_corr",
    "lidar_2020_DTM":                         "lidar_2020_DTM_ref",
    "lidar_2020_DSM":                         "lidar_2020_DSM_ref",
    "lidar_2024_dtm_filled_vref2020_rmdse":   "lidar_2024_DTM_corr",
    "lidar_2024_dsm_filled_vref2020_rmdse":   "lidar_2024_DSM_corr",
    "sfm_dtm_filled_vref2020":                "sfm_DTM_corr",
    "sfm_dsm_filled_vref2020":                "sfm_DSM_corr",
}

for old, new in rename_pairs.items():
    print(f"Renaming {old} -> {new}")
    gs.run_command("g.rename", raster=f"{old},{new}", overwrite=True)

# now your plotting list can be:
rasters = [
    "lidar_2017_DTM_corr",
    "lidar_2017_DSM_corr",
    "lidar_2020_DTM_ref",
    "lidar_2020_DSM_ref",
    "lidar_2024_DTM_corr",
    "lidar_2024_DSM_corr",
    "sfm_DTM_corr",
    "sfm_DSM_corr",
]
print(rasters)

In [ ]:
line_vec = "profile_test3"

rasters = [
    "lidar_2017_DTM_corr",
    "lidar_2017_DSM_corr",
    "lidar_2020_DTM_ref",
    "lidar_2020_DSM_ref",
    "lidar_2024_DTM_corr",
    "lidar_2024_DSM_corr",
    "sfm_DTM_corr",
    "sfm_DSM_corr",
]

# get line vertices as WKT, extract x y pairs
wkt_txt = gs.read_command("v.out.ascii", input=line_vec, format="wkt").strip()
wkt_line = wkt_txt.splitlines()[0]
pairs = re.findall(r"(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)", wkt_line)
coords = ",".join([f"{x},{y}" for x, y in pairs])

out_dir = Path("./profiles_out_vref2020")
out_dir.mkdir(parents=True, exist_ok=True)

for r in rasters:
    out_csv = out_dir / f"profile_{r}.csv"
    gs.run_command(
        "r.profile",
        input=r,
        coordinates=coords,
        resolution=1,
        output=str(out_csv),
        flags="g",
        overwrite=True,
    )
    print("Wrote:", out_csv)

In [ ]:
files = sorted(glob.glob("./profiles_out_vref2020/profile_*.csv"))
if not files: raise FileNotFoundError("No profile_*.csv found")

def load_profile(f):
    df = pd.read_csv(f, header=None, comment="#", sep=None, engine="python")
    if df.shape[1] == 1:
        df = df[0].astype(str).str.strip()
        df = df[(df != "") & (~df.str.startswith("#"))].str.split(r"[,\s]+", expand=True)
    df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")

    if df.shape[1] >= 3:  # x,y,value -> distance
        x, y, v = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,-1].to_numpy()
        d = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        return d, v
    return df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy()

plt.figure(figsize=(18, 6))
for f in files:
    d, v = load_profile(f)
    label = re.sub(r"^profile_|\.csv$", "", os.path.basename(f))
    plt.plot(d, v, label=label)

plt.xlabel("Distance (m)")
plt.ylabel("Elevation")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)
plt.savefig("profile_plot_after_vertical_correction_vref2020.png", dpi=300, bbox_inches="tight")
plt.show()